# 04 — Auditoría de características para nivel de riesgo

Este cuaderno contrasta el Top 30 del análisis previo con el perfil de ausencia de la base cruda y el diccionario CEM vigente. Su producto no es un ranking predictivo: es una lista auditable de variables que pueden, o no, pasar a la siguiente fase.

> Regla: una variable con alta importancia no se aprueba si está ausente masivamente, se registra después del momento de predicción, es parte directa de la valoración o introduce un riesgo de equidad sin justificación.

In [1]:
from pathlib import Path
import unicodedata
import pandas as pd

BASE_DIR = Path.cwd()
RANK_PATH = BASE_DIR / 'referencia_repositorio_top30_riesgo.csv'
PROFILE_PATH = BASE_DIR / 'salidas_eda' / 'perfil_variables_completo.csv'
DICT_PATH = BASE_DIR / 'diccionario_cem_estructurado.csv'
OUTPUT_DIR = BASE_DIR / 'salidas_modelado'
OUTPUT_DIR.mkdir(exist_ok=True)
for path in [RANK_PATH, PROFILE_PATH, DICT_PATH]:
    assert path.exists(), f'Falta: {path.name}'

def canonizar(valor):
    valor = unicodedata.normalize('NFKD', str(valor)).encode('ascii', 'ignore').decode('ascii')
    return valor.strip().upper()

rank = pd.read_csv(RANK_PATH)
profile = pd.read_csv(PROFILE_PATH)
dictionary = pd.read_csv(DICT_PATH)
rank['clave'] = rank['feature'].map(canonizar)
profile['clave'] = profile['columna_original'].map(canonizar)
dictionary['clave'] = dictionary['variable'].map(canonizar)

## Reglas de auditoría

Las reglas no declaran causalidad. Clasifican riesgo metodológico para decidir qué variables requieren validación de momento de registro, cuáles no deben pasar a un modelo prospectivo y cuáles se analizarán como contexto/fairness en vez de predictor individual.

In [2]:
audit = rank.merge(profile[['clave', 'pct_nulos', 'categorias_observadas']], on='clave', how='left')
audit = audit.merge(dictionary[['clave', 'variable', 'nombre_variable', 'obligatoria', 'categorias_documentadas', 'descripcion_documentada']], on='clave', how='left')

POSTERIORES = {'DESEA_DENUNCIAR', 'DESEA_PATROCINIO_LEGAL', 'SITUACION_AGRESOR'}
GEOGRAFIA_SERVICIO = {'CEM', 'UBIGEO_CODIGO', 'UBIGEO_NOMBRE', 'DPTO_DOMICILIO', 'PROV_DOMICILIO', 'DIST_DOMICILIO'}
COMPONENTES_RIESGO = {
    'AMENAZA_DE_MUERTE', 'FACTOR_AGRESOR_CONSUMO_ALCOHOL', 'FACTOR_AGRESOR_CONSUME_DROGA',
    'PRIMERA_VEZ_AGREDE', 'FRECUENCIA_AGREDE', 'ESTADO_AGRESOR_G', 'GRITOS_INSULTOS',
    'DESVALORIZACION_HUMILLACION', 'VIGILANCIA_CONTINUA_PERSECUCION', 'OTRA_VPSI',
    'JALONES_CABELLO', 'EMPujONES'.upper(), 'AHORCAMIENTO', 'NEGLIGENCIA', 'VIOLACION',
    'VINCULO_AFECTIVO_NINGUNO', 'PUNETAZOS',
}
DERIVADAS = {'ANIO_MES', 'UBIGEO_CODIGO', 'UBIGEO_NOMBRE'}

def clasificar(fila):
    clave = fila['clave']
    if clave in POSTERIORES:
        return 'BLOQUEADA_POSIBLE_RESULTADO_POSTERIOR'
    if clave == 'CONDICION':
        return 'REVISAR_FUGA_ADMINISTRATIVA'
    if clave in GEOGRAFIA_SERVICIO:
        return 'REVISAR_EQUIDAD_Y_GENERALIZACION'
    if clave in COMPONENTES_RIESGO:
        return 'REVISAR_COMPONENTE_DE_VALORACION'
    if clave in DERIVADAS:
        return 'DOCUMENTAR_FEATURE_DERIVADA'
    if pd.isna(fila['nombre_variable']):
        return 'REVISAR_NOMBRE_O_DICCIONARIO'
    if fila['pct_nulos'] > 20:
        return 'REQUIERE_REGLA_DE_AUSENCIA'
    return 'CANDIDATA_AUDITABLE'

audit['estado_inicial'] = audit.apply(clasificar, axis=1)
audit['regla'] = audit['estado_inicial'].map({
    'BLOQUEADA_POSIBLE_RESULTADO_POSTERIOR': 'Excluir del escenario prospectivo.',
    'REVISAR_FUGA_ADMINISTRATIVA': 'Confirmar si se conoce antes de la valoración y si no codifica una decisión de servicio.',
    'REVISAR_EQUIDAD_Y_GENERALIZACION': 'Usar para auditoría territorial; no incluir por defecto en decisión individual.',
    'REVISAR_COMPONENTE_DE_VALORACION': 'Permitida solo en réplica retrospectiva si se declara que reproduce la valoración.',
    'DOCUMENTAR_FEATURE_DERIVADA': 'Definir fórmula y ajustar la transformación solo con entrenamiento.',
    'REQUIERE_REGLA_DE_AUSENCIA': 'Definir el significado de nulo antes de transformar.',
    'REVISAR_NOMBRE_O_DICCIONARIO': 'Resolver correspondencia antes de usar.',
    'CANDIDATA_AUDITABLE': 'Confirmar momento de registro y codificación.',
})
audit = audit.sort_values('score_consenso', ascending=False)
audit.to_csv(OUTPUT_DIR / 'auditoria_top30_previa_nivel_riesgo.csv', index=False, encoding='utf-8-sig')
display(audit[['feature', 'score_consenso', 'pct_nulos', 'obligatoria', 'estado_inicial', 'regla']])
display(audit['estado_inicial'].value_counts().rename_axis('estado').reset_index(name='variables'))

,feature,score_consenso,pct_nulos,obligatoria,estado_inicial,regla
0,ubigeo_nombre,0.993307,NaN,NaN,REVISAR_EQUIDAD_Y_GENERALIZACION,Usar para auditoría territorial; no incluir po...
1,cem,0.981063,0.00,Si,REVISAR_EQUIDAD_Y_GENERALIZACION,Usar para auditoría territorial; no incluir po...
2,ubigeo_codigo,0.963858,NaN,NaN,REVISAR_EQUIDAD_Y_GENERALIZACION,Usar para auditoría territorial; no incluir po...
3,factor_agresor_consumo_alcohol,0.962835,77.84,No,REVISAR_COMPONENTE_DE_VALORACION,Permitida solo en réplica retrospectiva si se ...
4,amenaza_de_muerte,0.916732,90.95,No,REVISAR_COMPONENTE_DE_VALORACION,Permitida solo en réplica retrospectiva si se ...
5,factor_agresor_consume_droga,0.910472,94.43,No,REVISAR_COMPONENTE_DE_VALORACION,Permitida solo en réplica retrospectiva si se ...
6,violacion,0.803268,92.88,No,REVISAR_COMPONENTE_DE_VALORACION,Permitida solo en réplica retrospectiva si se ...
7,negligencia,0.802953,95.88,No,REVISAR_COMPONENTE_DE_VALORACION,Permitida solo en réplica retrospectiva si se ...
8,punetazos,0.769567,86.73,No,REVISAR_COMPONENTE_DE_VALORACION,Permitida solo en réplica retrospectiva si se ...
9,desea_denunciar,0.741339,78.21,No,BLOQUEADA_POSIBLE_RESULTADO_POSTERIOR,Excluir del escenario prospectivo.


,estado,variables
0,REVISAR_COMPONENTE_DE_VALORACION,17
1,REVISAR_EQUIDAD_Y_GENERALIZACION,6
2,BLOQUEADA_POSIBLE_RESULTADO_POSTERIOR,3
3,REQUIERE_REGLA_DE_AUSENCIA,2
4,CANDIDATA_AUDITABLE,1
5,DOCUMENTAR_FEATURE_DERIVADA,1


## Decisión de paso

Para la **réplica retrospectiva**, se puede evaluar el bloque de componentes de valoración, pero se reportará como capacidad de reproducir una etiqueta registrada. Para el modelo **prospectivo**, solo pasan variables aprobadas tras confirmar que anteceden a la valoración y tienen una regla de ausencia reproducible. La selección híbrida/MOES se repetirá exclusivamente en 2020–2023; 2024 y 2025 permanecen fuera de toda selección.